# 선정 계열 모델 비교 (참고) — 그 외 추가 실험 기록

3-3/4-3(계절차분 검증) 이후 대화 중 추가로 시도한 실험들을 정리한다. 전부
X3_l1(X3) 기준 스팟체크이며, 어느 노트북에도 기록 안 돼 있어서
여기 참고용으로만 모아둔다. 목적과 결과 위주로 간단히 정리(코드 재실행 없이
이미 확인된 실수치만 표로 재구성).

## 1. GARCH(1,1)/EGARCH/GJR-GARCH — Auto-ARIMA 위에 레이어드

**목적**: GARCH(1,1) 외에 비대칭 변동성(레버리지 효과)을 잡는 EGARCH·
GJR-GARCH도 시도하면 나아지는지 확인.

In [1]:
import pandas as pd

df1 = pd.DataFrame([
    ("정적(static, 변동성모델 없음)", 0.5756, -7.416),
    ("ARIMA+GARCH(1,1)", 0.5721, -7.300),
    ("ARIMA+GJR-GARCH", 0.5799, -7.343),
    ("ARIMA+EGARCH", 0.6928, -55.578),
], columns=["모델", "CRPS", "LL"]).set_index("모델").sort_values("CRPS")
display(df1)


,CRPS,LL
모델,,
"ARIMA+GARCH(1,1)",0.5721,-7.300
"정적(static, 변동성모델 없음)",0.5756,-7.416
ARIMA+GJR-GARCH,0.5799,-7.343
ARIMA+EGARCH,0.6928,-55.578


**결과**: GARCH(1,1)만 근소하게 정적보다 나음(노이즈 수준). GJR-GARCH는
더 나쁨. **EGARCH는 LL이 -55까지 폭락** — 60분기짜리 짧은 시계열에 비선형
로그분산 모델을 얹으면 수치적으로 불안정해진다는 걸 확인.

## 2. GARCH/SV/EGARCH/GJR-GARCH — 단독 사용(1차차분 기준)

**목적**: 변동성모델들을 Auto-ARIMA 없이 자체 평균방정식으로 단독 사용하면
어떤지 확인(1차차분만 적용한 z에 직접 적합).

In [2]:
df2 = pd.DataFrame([
    ("1차차분만 + 정적(std)", 0.5840, -7.330),
    ("1차차분만 + GARCH(1,1)", 0.5993, -7.286),
    ("1차차분만 + GJR-GARCH", 0.6061, None),
    ("1차차분만 + SV", 0.6295, -9.721),
    ("1차차분만 + EGARCH", 8.9986e+64, -61.746),
], columns=["모델", "CRPS", "LL"]).set_index("모델").sort_values("CRPS")
display(df2)


,CRPS,LL
모델,,
1차차분만 + 정적(std),5.840000e-01,-7.330
"1차차분만 + GARCH(1,1)",5.993000e-01,-7.286
1차차분만 + GJR-GARCH,6.061000e-01,NaN
1차차분만 + SV,6.295000e-01,-9.721
1차차분만 + EGARCH,8.998600e+64,-61.746


**결과**: 전부 정적보다 나쁨. **EGARCH는 단독 사용 시 CRPS가 사실상
무한대(9×10⁶⁴) 수준으로 완전히 발산** — 레이어드보다도 훨씬 불안정하다.

## 3. 차분 구조 on/off × 변동성모델 (2x2, X3_l1)

**목적**: 무차분/1차차분/계절차분/1차+계절차분 4가지 조합 각각에 정적·
GARCH(1,1)·SV를 적용해 어느 조합이 그나마 나은지 전수 확인.

In [3]:
df3 = pd.DataFrame([
    ("1차차분만 + 정적(std)", 0.5840, -7.330),
    ("1차차분만 + GARCH(1,1)", 0.5993, -7.286),
    ("1차차분만 + SV", 0.6295, -9.721),
    ("계절차분만 + 정적(std)", 0.7430, -8.400),
    ("계절차분만 + GARCH(1,1)", 0.7451, -8.442),
    ("무차분 + GARCH(1,1)", 0.8250, -7.878),
    ("계절차분만 + SV", 0.8378, -11.785),
    ("무차분 + SV", 0.8413, -9.019),
    ("무차분 + 정적(std)", 0.8486, -8.036),
    ("1차+계절차분 + 정적(std)", 0.9790, -8.770),
    ("1차+계절차분 + GARCH(1,1)", 1.0874, -9.009),
    ("1차+계절차분 + SV", 1.1687, -15.482),
], columns=["모델", "CRPS", "LL"]).set_index("모델").sort_values("CRPS")
display(df3)


,CRPS,LL
모델,,
1차차분만 + 정적(std),0.5840,-7.330
"1차차분만 + GARCH(1,1)",0.5993,-7.286
1차차분만 + SV,0.6295,-9.721
계절차분만 + 정적(std),0.7430,-8.400
"계절차분만 + GARCH(1,1)",0.7451,-8.442
"무차분 + GARCH(1,1)",0.8250,-7.878
계절차분만 + SV,0.8378,-11.785
무차분 + SV,0.8413,-9.019
무차분 + 정적(std),0.8486,-8.036


**결과**: **1차차분만 단독으로(변동성모델 없이) 쓰는 게 12개 조합 중
1위.** 계절차분은 어떤 조합으로도 확실히 나쁘고, 1차+계절차분 동시 적용은
과차분(over-differencing)으로 최하위권.

## 4. 월별 데이터로 표본 확장 (X2, FRED 실제 원본)

**목적**: "표본(60분기)이 부족해서 안 되는 건가?"를 확인하려고 X2
금리(FRED DFII10)의 실제 월별 원본을 받아와 origin을 36개→193개로
5배 이상 늘려서 재검증.

In [4]:
df4 = pd.DataFrame([
    ("기존(분기 전용, origin 36개)", 0.5467, 0.0293),
    ("월별기반(origin 193개, 분기 환산)", 0.5498, 0.0321),
], columns=["방식", "CRPS", "h=4 90%구간 폭"]).set_index("방식")
display(df4)
print("폭 비율(월별/분기전용) = 1.09x")


,CRPS,h=4 90%구간 폭
방식,,
"기존(분기 전용, origin 36개)",0.5467,0.0293
"월별기반(origin 193개, 분기 환산)",0.5498,0.0321


폭 비율(월별/분기전용) = 1.09x


**결과**: CRPS·구간폭 모두 **거의 동일**(차이 9% 이내, 노이즈 수준) —
calib 표본을 5배 늘려도 구간이 안 좁혀짐. **표본 부족이 원인이 아니라
변수 자체가 원래 예측하기 어렵다(거의 랜덤워크)**는 뜻.

## 5. Quantile Regression (직접 h=4 학습, X3_l1)

**목적**: 점예측+잔차 리샘플링 대신, 5%/95% 지점을 각각 별도 모델로 직접
학습(pinball loss)하면 더 타이트한 구간이 나오는지 확인. `lag=2` 선형
QuantReg으로 h=4 타겟을 직접 예측. **E3와 동등 비교를 위해 커버리지
90%에 맞춰 scale을 튜닝한 뒤 폭을 비교한다**(튜닝 전 비교는 불공정하다는
지적을 반영).

In [5]:
df5 = pd.DataFrame([
    ("QR 원본(튜닝 없음, scale=1.0)", 37, "40.5%", 351.7),
    ("QR 튜닝 후(scale=4.5, 90%에 맞춤)", 37, "89.2%", 1582.8),
    ("E3 앙상블(scale=1.42)", 22, "90.9%", 1207.7),
], columns=["방법", "origin 수", "커버리지", "평균 폭"]).set_index("방법")
display(df5)
print("튜닝 후에도 하한이 음수인 origin 비율: 64.9%")


,origin 수,커버리지,평균 폭
방법,,,
"QR 원본(튜닝 없음, scale=1.0)",37,40.5%,351.7
"QR 튜닝 후(scale=4.5, 90%에 맞춤)",37,89.2%,1582.8
E3 앙상블(scale=1.42),22,90.9%,1207.7


튜닝 후에도 하한이 음수인 origin 비율: 64.9%


**결과**: 커버리지를 공정하게 90%로 맞추면 **QR의 폭(1582.8)이 E3(1207.7)
보다 오히려 더 넓다** — 원래(튜닝 전) "QR이 더 좁아 보인다"던 건 커버리지를
안 맞춘 상태의 착시였다.

더 심각한 문제: scale=4.5까지 키워 커버리지를 맞췄는데도 **하한이 음수인
origin이 64.9%**(X3은 음수 불가능한데도) — 단순히 덜 좁혀서가
아니라 **중심값 자체가 2022~2023년 급등기를 반영 못 해 구조적으로 잘못된
방향**을 가리키고 있어서, 아무리 넓혀도 근본적으로 안 맞는다. Quantile
Regression 기각 결론은 유지되지만 이제 커버리지를 공정하게 맞춘 비교로
뒷받침된다.

## 6. 강건(robust) 캘리브레이션 — 극단잔차 trim

**목적**: 캘리브레이션 풀에서 절댓값이 큰 극단 잔차를 1~2개 제외하면
"평상시" 구간이 좁아지는지 확인(E3 앙상블, X3_l1, h=4).

In [6]:
df6 = pd.DataFrame([
    ("baseline(trim 없음)", 1.48, "90.9%", 1264.8),
    ("상위 1개 trim", 1.92, "90.9%", 1475.1),
    ("상위 2개 trim", 1.76, "90.9%", 1228.2),
], columns=["방법", "scale", "커버리지", "평균 폭"]).set_index("방법")
display(df6)


,scale,커버리지,평균 폭
방법,,,
baseline(trim 없음),1.48,90.9%,1264.8
상위 1개 trim,1.92,90.9%,1475.1
상위 2개 trim,1.76,90.9%,1228.2


**결과**: 상위 1개 trim은 **오히려 역효과**(폭 1265→1475) — 캘리브레이션
풀에서 극단치를 빼도 테스트셋의 실제 극단치는 그대로라 커버리지를
맞추려면 scale을 더 키워야 했다. 상위 2개 trim은 baseline과 거의
동일(노이즈 수준).

## 7. 적응형(adaptive) 폭 — 최근성 가중

**목적**: "최근 오차가 크면 이번 구간만 배율을 키운다"는 단순 휴리스틱이
GARCH보다 나은지 확인(threshold=1.5x, mult=1.3x).

In [7]:
df7 = pd.DataFrame([
    ("scale=1.00 (최적 근접)", "86.4%(미달)", 1075.2),
    ("scale=1.50 (커버리지 확보)", "95.5%(과도)", 1612.9),
], columns=["설정", "커버리지", "평균 폭"]).set_index("설정")
display(df7)
print("신호(적응 배율) 발동 비율: 86.4% -- 사실상 거의 항상 켜짐")


,커버리지,평균 폭
설정,,
scale=1.00 (최적 근접),86.4%(미달),1075.2
scale=1.50 (커버리지 확보),95.5%(과도),1612.9


신호(적응 배율) 발동 비율: 86.4% -- 사실상 거의 항상 켜짐


**결과**: 신호가 86.4% 확률로 거의 항상 켜져서 "평소엔 좁게, 위험할
때만 넓게"라는 원래 취지가 무색해짐. origin이 22개뿐이라 커버리지가
86.4%↔95.5%로 건너뛰어 정확히 90%를 못 맞춤. **근본 원인**: 이 휴리스틱도
GARCH와 같은 "최근 변동성으로 미래를 짐작한다"는 전제를 깔고 있는데,
X3_l1의 문제는 예고 없는 점프(X3 매입 결정)라 원리적으로 대비가
안 됨.

## 참고: 논의만 하고 테스트 안 한 것들

- **Heston / SABR / Stein-Stein / LSV**: 전부 옵션시장(내재변동성 곡면)
  전제 모델이라 우리 데이터엔 개념적으로 안 맞아 테스트 보류.
- **외생변수(exogenous) 도입**: "공용모델 목적"이라 보류.
- **롤링 1분기 예측(h=1을 4번 갱신)**: "4분기 고정 예측이 목표"라 보류.
- **SARIMAX(ARIMA+외생변수)**: 외생변수 아이디어를 `pmdarima`의 `X=`
  파라미터로 저비용 구현하는 방안으로 제안, 아직 미착수.

## 결론

7가지 추가 실험(GARCH 변형 3종 레이어드/단독, 차분×변동성 12조합, 월별
데이터 확장, quantile regression, robust trim, adaptive 폭) **전부
E3 앙상블(scale=1.42) 기준선을 못 넘었다.** 특히 EGARCH는 레이어드·단독
둘 다 수치적으로 발산할 만큼 이 데이터(60분기)에 안 맞았고, quantile
regression은 커버리지 자체를 못 지켰다.

지금까지 총 12개 넘는 독립적 방법이 실패한 것과 종합하면, **구간을
더 좁히는 시도는 여기서 멈추고 "이 정도가 정직한 한계"로 문서화하는 게
맞다**는 기존 결론이 다시 한번 확인된다.